# BIST 100 Preprocessing

This notebook converts validated BIST 100 history into model-ready arrays for one-step-ahead closing-value prediction.

The pipeline applies three safeguards:

1. observations are split chronologically without shuffling;
2. scaling parameters are learned from the training period only;
3. every target remains inside its assigned split and uses only earlier observations as input.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from bist100_forecasting.data import DEFAULT_DATA_PATH, load_history
from bist100_forecasting.preprocessing import (
    DEFAULT_LOOKBACK,
    DEFAULT_PROCESSED_DATA_PATH,
    create_windowed_split,
    fit_scalers,
    save_windowed_split,
    split_history,
)

pd.options.display.float_format = "{:,.4f}".format
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / DEFAULT_DATA_PATH
OUTPUT_PATH = PROJECT_ROOT / DEFAULT_PROCESSED_DATA_PATH

In [ ]:
history = load_history(DATA_PATH)
print(f"Loaded {len(history):,} validated observations from {DATA_PATH}")
history.tail()

## Chronological split

The oldest 70% of observations form the training period, the following 15% form the validation period, and the newest observations form the test period.

In [ ]:
split = split_history(history)
split_periods = pd.DataFrame(
    [
        {
            "Split": name,
            "Rows": len(frame),
            "First date": frame.index.min().date(),
            "Last date": frame.index.max().date(),
        }
        for name, frame in (
            ("Train", split.train),
            ("Validation", split.validation),
            ("Test", split.test),
        )
    ]
).set_index("Split")
split_periods

## Training-only scaling

Feature and target Min-Max scalers are fitted only on the training period. Values outside the training range may therefore be below 0 or above 1 in later periods; this is expected and confirms that those periods were not used to refit the scalers.

In [ ]:
scalers = fit_scalers(split.train)
training_ranges = pd.DataFrame(
    {
        "Feature": scalers.feature_columns,
        "Training minimum": scalers.feature_scaler.data_min_,
        "Training maximum": scalers.feature_scaler.data_max_,
    }
).set_index("Feature")
training_ranges

In [ ]:
scaled_ranges = pd.DataFrame(
    [
        {
            "Split": name,
            "Scaled minimum": scaled.min(),
            "Scaled maximum": scaled.max(),
        }
        for name, scaled in (
            ("Train", scalers.transform_features(split.train)),
            ("Validation", scalers.transform_features(split.validation)),
            ("Test", scalers.transform_features(split.test)),
        )
    ]
).set_index("Split")
scaled_ranges

## Sixty-day sequence windows

Each input contains 60 trading days and five scaled OHLCV features. Its target is the scaled closing value for the immediately following trading day.

In [ ]:
windows = create_windowed_split(split, scalers, lookback=DEFAULT_LOOKBACK)
window_summary = pd.DataFrame(
    [
        {
            "Split": name,
            "Input shape": window.features.shape,
            "Target shape": window.targets.shape,
            "First target": window.target_dates.min().date(),
            "Last target": window.target_dates.max().date(),
        }
        for name, window in (
            ("Train", windows.train),
            ("Validation", windows.validation),
            ("Test", windows.test),
        )
    ]
).set_index("Split")
window_summary

In [ ]:
first_validation_date = windows.validation.target_dates[0]
restored_validation_target = scalers.inverse_target(
    np.asarray([windows.validation.targets[0]])
)[0]
pd.Series(
    {
        "Target date": first_validation_date.date(),
        "Window shape": windows.validation.features[0].shape,
        "Restored target": restored_validation_target,
        "Recorded close": history.loc[first_validation_date, "Close"],
    },
    name="First validation sample",
).to_frame()

## Save the processed arrays

The compressed archive contains all split arrays, target dates, column names, lookback length, and scaling parameters required by later modeling stages. It is generated locally and is not committed to Git.

In [ ]:
output_path = save_windowed_split(windows, scalers, OUTPUT_PATH)
print(f"Saved model-ready arrays to {output_path}")

with np.load(output_path, allow_pickle=False) as prepared:
    archive_summary = {
        name: prepared[name].shape
        for name in prepared.files
    }

pd.Series(archive_summary, name="Shape").to_frame()

## Next step

The prepared arrays can now be used to establish simple forecasting baselines and evaluation metrics before training the LSTM and GRU models.